# 01 - Optimized Data Ingestion (Final Plan)

**Goal**: Pull the **most relevant, high-signal fields** (not everything) so we can build a fast-loading but powerful panel to predict **which specific companies** will go bankrupt in year N (e.g. 2026).

Strategy:
- Use `SELECT` with only the best columns on each table
- Use real table names for this WRDS account (`dsf_v2`, `dsedelist`, `wrdsapps.firm_ratio`, etc.)
- Proper joining keys (`gvkey`, `permno`, `cik`, `ticker`)
- Create foundation for 1-year forward default target

**Next notebooks will turn this into a clean firm-year panel + 2026 predictions.**

In [9]:
import wrds, pandas as pd, numpy as np
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)
print('Ready.')

Ready.


In [10]:
def get_conn():
    u = 'jasminelem'
    try: return wrds.Connection(wrds_username=u)
    except: return wrds.Connection(wrds_username=u, wrds_password='$Rapsjazz11')

conn = get_conn()
print('Connected as', conn._username)

Loading library list...
Done
Connected as jasminelem


## 0. Quick Diagnostic (real tables you have)

In [4]:
print('CRSP stock tables:', conn.list_tables('crsp_a_stock')[:10])
print('wrdsapps ratio tables:', [t for t in conn.list_tables('wrdsapps') if 'ratio' in t][:4])

CRSP stock tables: ['dse', 'dseall', 'dsedelist', 'dsedist', 'dseexchdates', 'dsenames', 'dsenasdin', 'dseshares', 'dsf', 'dsf_v2']
wrdsapps ratio tables: ['firm_ratio', 'firm_ratio_ccm', 'firm_ratio_ibes', 'firm_ratio_ibes_ccm']


## 1. Compustat Fundamentals (Key high-signal columns only)

In [5]:
funda_cols = "gvkey, datadate, fyear, conm, cik, at, lt, sale, ni, oiadp, dp, re, rect, invt, ppent, che, act, lct, dlc, dltt, seq, xint, capx, prcc_f, csho"
q = f"SELECT {funda_cols} FROM comp_na_daily_all.funda WHERE consol='C' AND indfmt='INDL' AND datafmt='STD' AND at > 0 AND fyear BETWEEN 1990 AND 2025"
df_funda = conn.raw_sql(q)
df_funda['datadate'] = pd.to_datetime(df_funda['datadate'])
df_funda.to_parquet(DATA_RAW/'comp_funda_key.parquet', index=False)
print('funda (key cols):', len(df_funda), 'rows')
print('Cell 5a done — funda saved. Next cell pulls fundq in safe yearly chunks.')

funda (key cols): 329338 rows
fundq (key cols): 1655455 rows


: 

## 1b. Quarterly Fundamentals — chunked by year (safe, no kernel crash)

In [ ]:
# Chunked fundq pull — one year at a time (prevents long-running cancellation / OOM)
fundq_cols = "gvkey, datadate, fyearq, fqtr, niq, oiadpq, saleq, atq"
years = list(range(1990, 2026))
fundq_parts = []

for y in years:
    q = f"SELECT {fundq_cols} FROM comp_na_daily_all.fundq WHERE consol='C' AND indfmt='INDL' AND datafmt='STD' AND fyearq = {y}"
    try:
        part = conn.raw_sql(q)
        fundq_parts.append(part)
        print(f'  fundq year {y}: {len(part)} rows')
    except Exception as e:
        print(f'  Skipped year {y}:', e)

df_fundq = pd.concat(fundq_parts, ignore_index=True) if fundq_parts else pd.DataFrame()
df_fundq['datadate'] = pd.to_datetime(df_fundq['datadate'])
df_fundq.to_parquet(DATA_RAW/'comp_fundq_key.parquet', index=False)
print('fundq (chunked):', len(df_fundq), 'rows total — saved safely.')
print('Cell 5b done. Kernel stays responsive.')

## 2. Pre-computed Ratios (almost all — they are high signal)

In [ ]:
# Robust ratio pull with fallback (handles different WRDS schema versions)
try:
    ratio_q = "SELECT gvkey, adate AS datadate, bm, roa, roe, debt_ebitda, de_ratio, debt_at, curr_ratio, quick_ratio, cash_ratio, intcov_ratio FROM wrdsapps.firm_ratio WHERE adate >= '1990-01-01'"
    df_ratio = conn.raw_sql(ratio_q)
except Exception:
    # Fallback: pull only columns we know exist
    ratio_q = "SELECT gvkey, adate AS datadate, bm, roa, roe, debt_ebitda, de_ratio, curr_ratio FROM wrdsapps.firm_ratio WHERE adate >= '1990-01-01'"
    df_ratio = conn.raw_sql(ratio_q)
df_ratio['datadate'] = pd.to_datetime(df_ratio['datadate'])
df_ratio.to_parquet(DATA_RAW/'firm_ratios_key.parquet', index=False)
print('firm_ratio (key):', len(df_ratio), 'rows')

## 3. Linking Table (essential for joining gvkey ↔ permno)

In [ ]:
link_q = "SELECT gvkey, lpermno AS permno, linkdt, linkenddt, linktype, linkprim FROM crsp_a_ccm.ccmxpf_lnkhist WHERE linktype IN ('LU','LC') AND linkprim IN ('P','C')"
df_link = conn.raw_sql(link_q)
df_link['linkdt'] = pd.to_datetime(df_link['linkdt'])
df_link['linkenddt'] = pd.to_datetime(df_link['linkenddt']).fillna(pd.Timestamp('2262-04-11'))
df_link.to_parquet(DATA_RAW/'linkhist_key.parquet', index=False)
print('linking table:', len(df_link), 'rows')

## 4. CRSP Market Data + Delistings (high-signal only)

In [ ]:
# CRSP Daily Returns - use classic dsf (has correct columns 'date', 'prc', 'ret', etc.)
msf_q = "SELECT permno, date, prc, ret, vol, shrout FROM crsp_a_stock.dsf WHERE date >= '1995-01-01'"
df_msf = conn.raw_sql(msf_q)
df_msf['date'] = pd.to_datetime(df_msf['date'])
df_msf.to_parquet(DATA_RAW/'crsp_dsfv2_key.parquet', index=False)
print('CRSP returns (dsf key cols):', len(df_msf), 'rows')

# Delistings
del_q = "SELECT permno, dlstdt, dlstcd FROM crsp_a_stock.dsedelist WHERE dlstdt IS NOT NULL"
df_del = conn.raw_sql(del_q)
df_del['dlstdt'] = pd.to_datetime(df_del['dlstdt'])
df_del['bankrupt_delist'] = df_del['dlstcd'].between(500, 599).astype(int)
df_del.to_parquet(DATA_RAW/'delist_key.parquet', index=False)
print('delistings:', len(df_del), 'rows')

## 5a. FJC Bankruptcy Filings

In [4]:
# FJC litigation bankruptcy (single table, usually manageable)
try:
    df_fjc = conn.raw_sql("SELECT * FROM fjc_litigation.bankruptcy WHERE filedate >= '1995-01-01'")
    df_fjc.to_parquet(DATA_RAW/'fjc_bankruptcy.parquet', index=False)
    print('FJC bankruptcy:', len(df_fjc))
except Exception as e: print('FJC error:', str(e)[:120])
print('5a FJC done.')

KeyboardInterrupt: 

## 5b. CIQ Key Developments (bankruptcy-related) — year-by-year

In [13]:
# === CIQ Key Developments — Ultra-safe pull ===
# Pulls only the columns needed (gvkey + announcedate + keydeveventtypeid)
# Avoids the huge 'situation' column completely.
import pandas as pd
from pathlib import Path
import time
print('Starting safe CIQ pull (10-40 min expected)...')
all_ciq = []
target_ids = (192, 26, 74, 31, 83)   # bankruptcy + going_concern + auditor_change + restatement

for yr in range(1995, 2026):
    for attempt in range(2):   # full year then half-year fallback
        try:
            if attempt == 0:
                q = f"SELECT gvkey, announcedate, keydeveventtypeid, eventtype, headline FROM ciq_keydev.wrds_keydev WHERE announcedate >= '{yr}-01-01' AND announcedate < '{yr+1}-01-01' AND keydeveventtypeid IN {target_ids}"
            else:
                m1 = 1 + (attempt-1)*6; m2 = 6 + (attempt-1)*6
                q = f"SELECT gvkey, announcedate, keydeveventtypeid, eventtype, headline FROM ciq_keydev.wrds_keydev WHERE announcedate >= '{yr}-{m1:02d}-01' AND announcedate < '{yr+1 if m2==12 else yr}-{ (m2%12)+1 :02d}-01' AND keydeveventtypeid IN {target_ids}"
            tmp = conn.raw_sql(q)
            if len(tmp) > 0: all_ciq.append(tmp)
            print(f'  {yr} (try {attempt+1}): {len(tmp)} rows')
            break
        except Exception as e:
            print(f'  {yr} try {attempt+1} failed — will retry split if needed')
            time.sleep(1)

if all_ciq:
    df = pd.concat(all_ciq, ignore_index=True).drop_duplicates()
    df['announcedate'] = pd.to_datetime(df['announcedate'], errors='coerce')
    df['gvkey'] = df['gvkey'].astype(str)
    df.to_parquet(DATA_RAW/'ciq_keydev.parquet', index=False)
    print(f'\n✅ CIQ saved: {len(df):,} rows')
else:
    print('No CIQ data pulled.')
print('CIQ step finished.')


Starting safe CIQ pull (10-40 min expected)...
  1995 (try 1): 7082 rows
  1996 (try 1): 10474 rows
  1997 (try 1): 14306 rows
  1998 (try 1): 22987 rows
  1999 (try 1): 45308 rows
  2000 (try 1): 79281 rows
  2001 (try 1): 47693 rows
  2002 (try 1): 44874 rows
  2003 (try 1): 53374 rows
  2004 (try 1): 83809 rows
  2005 (try 1): 91774 rows
  2006 (try 1): 110123 rows
  2007 (try 1): 124627 rows
  2008 (try 1): 118301 rows
  2009 (try 1): 111947 rows
  2010 (try 1): 145204 rows
  2011 (try 1): 163900 rows


## 5c. Macro Rates (FED / FRB)

In [3]:
# Macro — small table
try:
    df_macro = conn.raw_sql("SELECT date, aaa, baa, fedfunds, gs10 FROM frb_all.rates_monthly WHERE date >= '1990-01-01'")
    df_macro.to_parquet(DATA_RAW/'macro_key.parquet', index=False)
    print('Macro:', len(df_macro))
except Exception as e: print('Macro error:', str(e)[:80])
print('5c Macro done.')

Macro: 422
5c Macro done.


## 5d. IBES Analyst Estimates — year-by-year

In [4]:
# IBES — chunked by year (already safe). Run independently.
try:
    all_ibes = []
    for yr in range(1995, 2026):
        q = f"SELECT ticker, statpers, fpi, meanest, stdev, numest FROM tr_ibes.statsum_epsus WHERE fpi IN ('1','6') AND statpers >= '{yr}-01-01' AND statpers < '{yr+1}-01-01'"
        tmp = conn.raw_sql(q)
        if len(tmp) > 0: all_ibes.append(tmp)
    df_ibes = pd.concat(all_ibes, ignore_index=True) if all_ibes else pd.DataFrame()
    df_ibes.to_parquet(DATA_RAW/'ibes_key.parquet', index=False)
    print('IBES key:', len(df_ibes))
except Exception as e: print('IBES error:', str(e)[:140])
print('5d IBES done — saved.')

IBES key: 3337765
5d IBES done — saved.


## 6. Master Linked Panel (handled by script)


In [6]:
print("=== Master Linked Panel Creation ===")
print("This step is now fully handled by the script:")
print("   python src/build_clean_training_data.py")
print()
print("The script performs:")
print("  - Strict AGENTS.md-compliant linking (linktype/linkprim + date filter)")
print("  - FJC + CIQ + Delist bankruptcy merging with correct priority")
print("  - Target creation + all rich features")
print("  - Time-based train/val/test splits")
print()
print("Just run the script after this notebook. It will create linked_master_panel.parquet automatically.")



=== Creating Master Linked Panel (loading from raw parquets) ===
Loaded funda: 329,338 | link: 33,324 | ratios: 1,912,037 | delist: 38,872
After time-valid link merge + dedup: 209498 rows
After attaching ratios: 1876637 rows
After attaching delist signals: 1876637 rows

✅ Master panel saved: 1876637 rows | 21135 companies
Ready for notebook 03 (forward target) and notebook 05/06 (modeling + 2026 predictions).


## Final Summary

In [7]:
print('='*78)
print('NOTEBOOK 01 COMPLETE — ALL RAW DATA PULLED')
print('='*78)

for fname in ['comp_funda_key.parquet','comp_fundq_key.parquet','firm_ratios_key.parquet',
              'linkhist_key.parquet','crsp_dsfv2_key.parquet','delist_key.parquet',
              'fjc_bankruptcy.parquet','fjc_wrds_link.parquet','ciq_keydev.parquet',
              'ibes_key.parquet','macro_key.parquet','macro_rates.parquet']:
    p = DATA_RAW / fname
    if p.exists():
        try:
            d = pd.read_parquet(p)
            print(f'✓ {fname:32} {len(d):>9,} rows')
        except:
            print(f'✓ {fname:32} (present)')
    else:
        print(f'✗ {fname:32} MISSING')

print('='*78)
print('NEXT STEP (single command):')
print('   python src/build_clean_training_data.py')
print()
print('This script will:')
print('  • Build the master linked panel (AGENTS.md compliant)')
print('  • Create default_in_next_12m target')
print('  • Add all rich features + CRSP market signals + CIQ flags')
print('  • Produce final monthly_train / val / test')
print('='*78)
conn.close()


NOTEBOOK 01 COMPLETE — OPTIMIZED FOR PREDICTING SPECIFIC COMPANIES IN YEAR N
✓ comp_funda_key.parquet             329,338 rows ×  25 cols
✓ comp_fundq_key.parquet           1,655,455 rows ×   8 cols
✓ firm_ratios_key.parquet          1,912,037 rows ×  12 cols
✓ linkhist_key.parquet                33,324 rows ×   6 cols
✓ crsp_dsfv2_key.parquet           58,457,885 rows ×   6 cols
✓ delist_key.parquet                  38,872 rows ×   4 cols
✓ fjc_bankruptcy.parquet              30,000 rows ×  89 cols
✓ macro_key.parquet                      422 rows ×   5 cols
✓ ibes_key.parquet                 3,337,765 rows ×   6 cols
✓ linked_master_panel.parquet      1,876,637 rows ×  42 cols
Next: 02_feature_engineering.ipynb → 03_bankruptcy_labeling (create forward target) → 05_final_panel → 06 (2026 company predictions + SHAP)


## ✅ Notebook 01 Complete
Raw data pulled. Now run:
```bash
python src/build_clean_training_data.py
```

Then open `notebooks/02_xgboost_modeling_hpo_shap.ipynb`.
